In [14]:

from sklearn.metrics import precision_score, recall_score, f1_score

# Get predicted probabilities for elevated stress (class 1)
y_prob = model.predict_proba(X_test)[:, 1]

print("Threshold comparison:")
print("-" * 65)
print(f"{'Threshold':<12}{'Precision':<14}{'Recall':<12}{'F1-score':<12}")

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(
        y_test, y_pred_threshold, zero_division=0
    )
    recall = recall_score(
        y_test, y_pred_threshold, zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred_threshold, zero_division=0
    )

    print(
        f"{threshold:<12.2f}{precision:<14.3f}"
        f"{recall:<12.3f}{f1:<12.3f}"
    )


Threshold comparison:
-----------------------------------------------------------------
Threshold   Precision     Recall      F1-score    
0.30        0.784         0.795       0.789       
0.35        0.838         0.781       0.809       
0.40        0.903         0.767       0.830       
0.45        0.907         0.671       0.772       
0.50        0.939         0.630       0.754       
0.55        0.930         0.548       0.690       
0.60        0.926         0.342       0.500       


In [13]:

print("Current model:")
print(model)

print("\nModel parameters related to classification:")
for key, value in model.get_params().items():
    if "class_weight" in key or "imputer" in key.lower():
        print(f"{key}: {value}")


Current model:
Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42))])

Model parameters related to classification:
simpleimputer: SimpleImputer(strategy='median')
simpleimputer__add_indicator: False
simpleimputer__copy: True
simpleimputer__fill_value: None
simpleimputer__keep_empty_features: False
simpleimputer__missing_values: nan
simpleimputer__strategy: median
logisticregression__class_weight: balanced


In [12]:



# Step 1: Load data, handle missing values, train and evaluate

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Load labeled dataset
df = pd.read_csv("../data/processed/labeled_stress_windows.csv")

# Re-create binary labels
df["stress_class"] = (
    df["self_reported_stress"] >= 5
).astype(int)

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["stress_class"].value_counts().sort_index())

# Select physiological features
feature_columns = [
    "hr_mean", "hr_std", "hr_min", "hr_max",
    "eda_mean", "eda_std", "eda_min", "eda_max",
    "acc_mean_magnitude", "acc_std_magnitude", "acc_max_magnitude",
    "bvp_mean", "bvp_std", "bvp_min", "bvp_max"
]

# Convert infinite values to missing values
X = df[feature_columns].replace([np.inf, -np.inf], np.nan)
y = df["stress_class"]
groups = df["subject_id"]

# Remove features that are entirely missing
all_missing = X.columns[X.isna().all()].tolist()

if all_missing:
    print("\nRemoving entirely missing features:", all_missing)
    X = X.drop(columns=all_missing)

# Report remaining missing values
print("\nMissing values per feature:")
print(X.isna().sum()[X.isna().sum() > 0])

# Split by participant to reduce data leakage
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("\nTraining windows:", len(X_train))
print("Testing windows:", len(X_test))
print("Training participants:", groups.iloc[train_idx].nunique())
print("Testing participants:", groups.iloc[test_idx].nunique())

# Impute using training medians, then scale and train
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )
)

model.fit(X_train, y_train)

# Evaluate on held-out participants
y_pred = model.predict(X_test)

print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 3))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Low stress", "Elevated stress"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))


Dataset shape: (401, 34)

Class distribution:
stress_class
0    163
1    238
Name: count, dtype: int64

Missing values per feature:
bvp_mean    6
bvp_std     6
bvp_min     6
bvp_max     6
dtype: int64

Training windows: 308
Testing windows: 93
Training participants: 26
Testing participants: 7

Accuracy: 0.677

Classification report:
                 precision    recall  f1-score   support

     Low stress       0.39      0.85      0.53        20
Elevated stress       0.94      0.63      0.75        73

       accuracy                           0.68        93
      macro avg       0.66      0.74      0.64        93
   weighted avg       0.82      0.68      0.71        93


Confusion matrix:
[[17  3]
 [27 46]]


In [5]:
# Step 1: Make the binary stress labels consistent

labeled_df["stress_class"] = (
    labeled_df["self_reported_stress"] >= 5
).astype(int)

print("Updated stress-class distribution:")
print(labeled_df["stress_class"].value_counts().sort_index())

print("\nCheck ratings around the threshold:")
display(
    labeled_df[
        labeled_df["self_reported_stress"].isin([4.0, 4.5, 5.0, 5.5])
    ][["self_reported_stress", "stress_class"]]
    .drop_duplicates()
    .sort_values("self_reported_stress")
)

Updated stress-class distribution:
stress_class
0    163
1    238
Name: count, dtype: int64

Check ratings around the threshold:


,self_reported_stress,stress_class
4,4.0,0
0,4.5,0
50,5.0,1
71,5.5,1


In [4]:

import pandas as pd

# Load the labeled dataset created in Notebook 2
model_df = pd.read_csv("../data/processed/labeled_stress_windows.csv")

# Convert timestamps to datetime
model_df["window_start"] = pd.to_datetime(model_df["window_start"], utc=True)
model_df["window_end"] = pd.to_datetime(model_df["window_end"], utc=True)

# Create the binary prediction target
model_df["stress_class"] = (
    model_df["self_reported_stress"] >= 5
).astype(int)

# Display the class distribution
print("Dataset shape:", model_df.shape)

print("\nStress class mapping:")
print("0 = Low stress (rating <= 4)")
print("1 = Elevated stress (rating >= 5)")

print("\nClass distribution:")
print(model_df["stress_class"].value_counts().sort_index())

print("\nClass percentages:")
print(
    (model_df["stress_class"].value_counts(normalize=True).sort_index() * 100)
    .round(2)
)

display(
    model_df[
        ["subject_id", "protocol_stage", "self_reported_stress", "stress_class"]
    ].head(10)
)


Dataset shape: (401, 34)

Stress class mapping:
0 = Low stress (rating <= 4)
1 = Elevated stress (rating >= 5)

Class distribution:
stress_class
0    163
1    238
Name: count, dtype: int64

Class percentages:
stress_class
0    40.65
1    59.35
Name: proportion, dtype: float64


,subject_id,protocol_stage,self_reported_stress,stress_class
0,S01,Stroop,4.5,0
1,S01,Stroop,4.5,0
2,S01,Stroop,4.5,0
3,S01,Stroop,4.5,0
4,S01,TMCT,4.0,0
5,S01,TMCT,4.0,0
6,S01,TMCT,4.0,0
7,S01,TMCT,4.0,0
8,S01,Real Opinion,6.0,1
9,S01,Opposite Opinion,7.0,1


In [3]:

import pandas as pd

# Load the labeled dataset
labeled_df = pd.read_csv("../data/processed/labeled_stress_windows.csv")

print("Dataset shape:", labeled_df.shape)

print("\nStress score statistics:")
display(labeled_df["self_reported_stress"].describe())

print("\nStress score distribution:")
display(
    labeled_df["self_reported_stress"]
    .value_counts()
    .sort_index()
)

print("\nWindows by task:")
display(labeled_df["protocol_stage"].value_counts())

print("\nWindows per participant:")
display(labeled_df.groupby("subject_id").size().describe())

print("\nMissing values in important columns:")
print(
    labeled_df[
        ["subject_id", "protocol_stage", "self_reported_stress"]
    ].isnull().sum()
)


Dataset shape: (401, 33)

Stress score statistics:


count    401.000000
mean       5.172070
std        2.124045
min        1.000000
25%        4.000000
50%        5.000000
75%        7.000000
max       10.000000
Name: self_reported_stress, dtype: float64


Stress score distribution:


self_reported_stress
1.0     16
1.5      1
2.0     17
3.0     60
4.0     65
4.5      4
5.0     78
5.5      5
6.0     41
7.0     42
8.0     48
9.0     17
10.0     7
Name: count, dtype: int64


Windows by task:


protocol_stage
TMCT                225
Stroop               61
Subtract             42
Real Opinion         37
Opposite Opinion     36
Name: count, dtype: int64


Windows per participant:


count    33.000000
mean     12.151515
std       1.622521
min       9.000000
25%      11.000000
50%      12.000000
75%      13.000000
max      18.000000
dtype: float64


Missing values in important columns:
subject_id              0
protocol_stage          0
self_reported_stress    0
dtype: int64


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the labeled dataset created in Notebook 2
data_path = Path("../data/processed/labeled_stress_windows.csv")
df = pd.read_csv("../data/processed/labeled_stress_windows.csv")

# Re-create the stress labels after loading the saved CSV
df["stress_class"] = (
    df["self_reported_stress"] >= 5
).astype(int)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nStress class distribution:")
print(df["stress_class"].value_counts().sort_index())
print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (401, 34)
Columns: ['session_id', 'session_key', 'activity', 'window_id', 'window_start', 'window_end', 'hr_mean', 'hr_std', 'hr_min', 'hr_max', 'eda_mean', 'eda_std', 'eda_min', 'eda_max', 'acc_mean_magnitude', 'acc_std_magnitude', 'acc_max_magnitude', 'bvp_mean', 'bvp_std', 'bvp_min', 'bvp_max', 'hr_sample_count', 'eda_sample_count', 'acc_sample_count', 'bvp_sample_count', 'bvp_quality_flag', 'hr_review_flag', 'subject_id', 'protocol_stage', 'label_version', 'self_reported_stress', 'overlap_seconds', 'overlap_fraction', 'stress_class']

Stress class distribution:
stress_class
0    163
1    238
Name: count, dtype: int64
Dataset loaded successfully!
Shape: (401, 34)

Columns:
['session_id', 'session_key', 'activity', 'window_id', 'window_start', 'window_end', 'hr_mean', 'hr_std', 'hr_min', 'hr_max', 'eda_mean', 'eda_std', 'eda_min', 'eda_max', 'acc_mean_magnitude', 'acc_std_magnitude', 'acc_max_magnitude', 'bvp_mean', 'bvp_std', 'bvp_min', 'bvp_max', 'hr_sample_count', '

,dtype
session_id,str
session_key,str
activity,str
window_id,int64
window_start,str
window_end,str
hr_mean,float64
hr_std,float64
hr_min,float64
hr_max,float64



First 5 rows:


,session_id,session_key,activity,window_id,window_start,window_end,hr_mean,hr_std,hr_min,hr_max,...,bvp_sample_count,bvp_quality_flag,hr_review_flag,subject_id,protocol_stage,label_version,self_reported_stress,overlap_seconds,overlap_fraction,stress_class
0,S01,STRESS/S01,STRESS,26,2013-02-20 18:08:29+00:00,2013-02-20 18:08:59+00:00,58.741333,1.604163,56.52,61.38,...,1920,False,False,S01,Stroop,v1,4.5,27.0,0.900000,0
1,S01,STRESS/S01,STRESS,27,2013-02-20 18:08:59+00:00,2013-02-20 18:09:29+00:00,54.054333,1.200321,52.68,56.40,...,1920,False,False,S01,Stroop,v1,4.5,30.0,1.000000,0
2,S01,STRESS/S01,STRESS,28,2013-02-20 18:09:29+00:00,2013-02-20 18:09:59+00:00,52.634667,0.208114,52.32,52.95,...,1920,False,False,S01,Stroop,v1,4.5,30.0,1.000000,0
3,S01,STRESS/S01,STRESS,29,2013-02-20 18:09:59+00:00,2013-02-20 18:10:29+00:00,52.619333,0.243227,52.40,53.27,...,1920,False,False,S01,Stroop,v1,4.5,18.0,0.600000,0
4,S01,STRESS/S01,STRESS,44,2013-02-20 18:17:29+00:00,2013-02-20 18:17:59+00:00,60.567000,1.326092,58.22,62.65,...,1920,False,False,S01,TMCT,v1,4.0,22.0,0.733333,0
